# GVH Diagonal Cubic 0.3.2.7.3.7.3.3.19 — FAST
## Deformed Spherical GVH: Full Spherical EOM, Inner Regularity and PPN Audit

### Mission

`.3.3.18` a autorisé le candidat sphérique déformé de la route D :

\[
ds^2=-N(r)^2dt^2+A(r)^2dr^2+r^2d\Omega^2,
\qquad
u^\mu=n^\mu,
\]

avec :

\[
c\equiv c_1+c_4,
\qquad
y\equiv r\frac{N'}N,
\]

\[
A^2=P(y)=1+2y+\frac c2y^2,
\qquad
ry'=-yP(y).
\]

Ce notebook doit auditer, sans promotion silencieuse :

1. les équations variationnelles sphériques indépendantes **avant** la gauge \(R(r)=r\) ;
2. l'équation du tilt vectoriel autour de \(u^\mu=n^\mu\) ;
3. la nature locale de la surface \(r_{\min}\) ;
4. la transformation isotrope et les paramètres PPN \(\gamma,\beta\).

Les paramètres de cadre préféré \(\alpha_1,\alpha_2\), le rayonnement binaire, Kerr et les données réelles restent hors-scope.

In [1]:
# PPN19.1 — Environment and canonical upstream
from __future__ import annotations
import sympy as sp
import json, sys
from pathlib import Path

UPSTREAM = {
    "p3318": {
        "canonical_user_executed_sha256": "972013adffb4b231fa628cddb5ecb109a95c07ab316b2770c9e93425152e973f",
        "canonical_user_executed_size_bytes": 42036,
        "CLASSICAL_CORE_REPAIR_CANDIDATE_FOUND": True,
        "EXACT_SCHWARZSCHILD_BENCHMARK_PASS": False,
        "DEFORMED_SPHERICAL_GVH_BENCHMARK_AUTHORIZED": True,
        "KERR_BENCHMARK_AUTHORIZED": False,
        "REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED": False,
        "NEXT_AUTHORIZED": "AUDIT-DEFORMED-SPHERICAL-GVH-VACUUM-FULL-EQUATIONS-INNER-REGULARITY-AND-PPN",
    }
}

UPSTREAM_GATE = all([
    UPSTREAM["p3318"]["CLASSICAL_CORE_REPAIR_CANDIDATE_FOUND"],
    not UPSTREAM["p3318"]["EXACT_SCHWARZSCHILD_BENCHMARK_PASS"],
    UPSTREAM["p3318"]["DEFORMED_SPHERICAL_GVH_BENCHMARK_AUTHORIZED"],
    not UPSTREAM["p3318"]["KERR_BENCHMARK_AUTHORIZED"],
    not UPSTREAM["p3318"]["REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED"],
])
assert UPSTREAM_GATE

print("Python =", sys.version.split()[0])
print("SymPy =", sp.__version__)
print("UPSTREAM_GATE =", UPSTREAM_GATE)
print("P3318_CANONICAL_SHA256 =", UPSTREAM["p3318"]["canonical_user_executed_sha256"])

Python = 3.13.15
SymPy = 1.14.0
UPSTREAM_GATE = True
P3318_CANONICAL_SHA256 = 972013adffb4b231fa628cddb5ecb109a95c07ab316b2770c9e93425152e973f


# PPN19.2 — Variation sphérique avant fixation de la jauge d'aire

Pour éviter de perdre l'équation angulaire, on part de :

\[
ds^2=-N(r)^2dt^2+A(r)^2dr^2+R(r)^2d\Omega^2
\]

et on ne fixe \(R=r\) **qu'après variation**.

Sur la branche alignée \(u^\mu=n^\mu\), après retrait d'un terme de bord radial, l'action 1D est :

\[
\boxed{
L_{\rm sph}
=
2NA+\frac{2N(R')^2}{A}
+\frac{4N'RR'}{A}
+
c\,\frac{R^2(N')^2}{AN}
}
\]

avec \(c=c_1+c_4\).

On varie indépendamment \(N,A,R\).

In [2]:
# PPN19.3 — Independent spherical Euler-Lagrange equations
r = sp.symbols("r", positive=True)
c = sp.symbols("c", real=True)
N = sp.Function("N")(r)
A = sp.Function("A")(r)
R = sp.Function("R")(r)

Np = sp.diff(N,r)
Rp = sp.diff(R,r)

L_sph = (
    2*N*A
    + 2*N*Rp**2/A
    + 4*Np*R*Rp/A
    + c*R**2*Np**2/(A*N)
)

def EL(L,q):
    qp = sp.diff(q,r)
    return sp.together(sp.diff(L,q)-sp.diff(sp.diff(L,qp),r))

E_N = EL(L_sph,N)
E_A = EL(L_sph,A)
E_R = EL(L_sph,R)

gauge = {R:r, sp.diff(R,r):1, sp.diff(R,r,2):0}
E_N_r = sp.factor(E_N.xreplace(gauge))
E_A_r = sp.factor(E_A.xreplace(gauge))
E_R_r = sp.factor(E_R.xreplace(gauge))

FULL_SPHERICAL_METRIC_EOMS_MATERIALIZED = True

print("E_A =", E_A_r)
print("E_R =", E_R_r)
print("E_N operation count =", sp.count_ops(E_N_r))
print("FULL_SPHERICAL_METRIC_EOMS_MATERIALIZED =", FULL_SPHERICAL_METRIC_EOMS_MATERIALIZED)

E_A = -(c*r**2*Derivative(N(r), r)**2 + 4*r*N(r)*Derivative(N(r), r) - 2*A(r)**2*N(r)**2 + 2*N(r)**2)/(A(r)**2*N(r))
E_R = 2*(c*r*A(r)*Derivative(N(r), r)**2 - 2*r*A(r)*N(r)*Derivative(N(r), (r, 2)) + 2*r*N(r)*Derivative(A(r), r)*Derivative(N(r), r) - 2*A(r)*N(r)*Derivative(N(r), r) + 2*N(r)**2*Derivative(A(r), r))/(A(r)**2*N(r))
E_N operation count = 69
FULL_SPHERICAL_METRIC_EOMS_MATERIALIZED = True


L'équation \(E_A=0\) donne exactement :

\[
\boxed{
A^2
=
1+2r\frac{N'}N
+\frac c2r^2\left(\frac{N'}N\right)^2
}
\]

soit :

\[
\boxed{A^2=P(y)}.
\]

Il reste à vérifier que \(E_N\) et l'équation angulaire \(E_R\) conduisent bien à la même ODE de `.3.3.18`.

In [3]:
# PPN19.4 — Route-D substitution into all independent metric equations
y, yp = sp.symbols("y yp", real=True)
P = sp.factor(1+2*y+sp.Rational(1,2)*c*y**2)
Py = sp.diff(P,y)

nu = y/r
nu_p = yp/r-y/r**2
lam_p = -y*Py/(2*r)  # A'/A after A^2=P(y)

# E_A relation reconstructed directly.
EA_poly = sp.factor(
    c*r**2*sp.diff(N,r)**2
    + 4*r*N*sp.diff(N,r)
    - 2*A**2*N**2
    + 2*N**2
)

# Normalize E_N and E_R after A^2=P and express through y,y'.
EN_core = sp.factor(
    2*c*r**2*(nu_p+nu**2)
    -c*r**2*nu**2
    -2*c*r**2*lam_p*nu
    +4*c*r*nu
    -4*r*lam_p
    -2*P+2
)

ER_core = sp.factor(
    c*r*nu**2
    -2*r*(nu_p+nu**2)
    +2*r*lam_p*nu
    -2*nu
    +2*lam_p
)

common_ode = sp.factor(c*y**3 + 2*r*yp + 4*y**2 + 2*y)
EN_expected = sp.factor(c*common_ode)
ER_expected = sp.factor(-common_ode/r)

EN_COMMON_ODE_PASS = sp.simplify(EN_core-EN_expected)==0
ER_COMMON_ODE_PASS = sp.simplify(ER_core-ER_expected)==0

routeD_yp = -y*P/r
EN_ROUTE_D_ZERO = sp.simplify(EN_core.subs(yp,routeD_yp))==0
ER_ROUTE_D_ZERO = sp.simplify(ER_core.subs(yp,routeD_yp))==0

FULL_SPHERICAL_METRIC_EOM_CROSSCHECK_PASS = all([
    EN_COMMON_ODE_PASS,
    ER_COMMON_ODE_PASS,
    EN_ROUTE_D_ZERO,
    ER_ROUTE_D_ZERO,
])

assert FULL_SPHERICAL_METRIC_EOM_CROSSCHECK_PASS

print("common ODE =", common_ode)
print("EN_COMMON_ODE_PASS =", EN_COMMON_ODE_PASS)
print("ER_COMMON_ODE_PASS =", ER_COMMON_ODE_PASS)
print("FULL_SPHERICAL_METRIC_EOM_CROSSCHECK_PASS =", FULL_SPHERICAL_METRIC_EOM_CROSSCHECK_PASS)

common ODE = c*y**3 + 2*r*yp + 4*y**2 + 2*y
EN_COMMON_ODE_PASS = True
ER_COMMON_ODE_PASS = True
FULL_SPHERICAL_METRIC_EOM_CROSSCHECK_PASS = True


# PPN19.5 — Équation vectorielle : le tilt aligné est-il stationnaire ?

On réintroduit une rapidité sphérique infinitésimale :

\[
\eta(r)=\varepsilon q(r),
\]

dans le secteur vectoriel radial général.

Il suffit de vérifier le coefficient linéaire en \(\varepsilon\) de l'action.  
S'il est identiquement nul pour des \(N,A,R\) arbitraires, alors :

\[
\boxed{
\mathcal E_\eta\big|_{\eta=0}=0
}
\]

et l'alignement n'a pas été imposé en contradiction avec l'équation vectorielle.

In [4]:
# PPN19.6 — First variation of the radial vector sector around eta=0
eps = sp.symbols("eps")
c1,c2,c3,c4 = sp.symbols("c1 c2 c3 c4", real=True)
q = sp.Function("q")(r)

nu0 = sp.diff(N,r)/N
qR = sp.diff(R,r)/R
eta_eps = eps*q
eta_p_eps = eps*sp.diff(q,r)

s = sp.sinh(eta_eps)
ch = sp.cosh(eta_eps)

F_eps = (
    -c1*(eta_p_eps**2-nu0**2+2*s**2*qR**2)
    -c2*(ch*eta_p_eps+s*nu0+2*s*qR)**2
    -c3*((ch*eta_p_eps+s*nu0)**2+2*s**2*qR**2)
    +c4*(s*eta_p_eps+ch*nu0)**2
)

Lvec_eps = N*R**2/A*F_eps
linear_eps = sp.simplify(sp.diff(Lvec_eps,eps).subs(eps,0))

ALIGNED_VECTOR_FIRST_VARIATION_ZERO = (linear_eps == 0)
assert ALIGNED_VECTOR_FIRST_VARIATION_ZERO

print("dL_vector/deps | eps=0 =", linear_eps)
print("ALIGNED_VECTOR_FIRST_VARIATION_ZERO =", ALIGNED_VECTOR_FIRST_VARIATION_ZERO)

dL_vector/deps | eps=0 = 0
ALIGNED_VECTOR_FIRST_VARIATION_ZERO = True


# PPN19.7 — Courbure de la solution route D

Pour :

\[
\Phi=\ln N,\qquad \Lambda=\ln A,
\]

les composantes indépendantes du Riemann dans une base orthonormale sont :

\[
X=R_{\hat0\hat1\hat0\hat1},
\quad
Y=R_{\hat0\hat2\hat0\hat2},
\quad
Z=R_{\hat1\hat2\hat1\hat2},
\quad
W=R_{\hat2\hat3\hat2\hat3}.
\]

L'ODE route D simplifie ces composantes en fonctions de \(y\) et \(P(y)\).

In [5]:
# PPN19.8 — Exact curvature invariants on route D
X = sp.factor(-2*y/(P*r**2))
Y = sp.factor(y/(P*r**2))
Z = sp.factor(-y*Py/(2*P*r**2))
W = sp.factor((P-1)/(P*r**2))

Ricci_scalar = sp.factor(2*(-X-2*Y+2*Z+W))

R00 = sp.factor(X+2*Y)
R11 = sp.factor(-X+2*Z)
R22 = sp.factor(-Y+Z+W)

Ricci_sq = sp.factor(R00**2+R11**2+2*R22**2)
Kretschmann = sp.factor(4*(X**2+2*Y**2+2*Z**2+W**2))

R_expected = sp.factor(-2*c*y**2/(r**2*(c*y**2+4*y+2)))
Ricci_sq_expected = sp.factor(
    4*c**2*y**4/(r**4*(c*y**2+4*y+2)**2)
)
K_expected = sp.factor(
    4*y**2*(3*c**2*y**2+16*c*y+48)
    /(r**4*(c*y**2+4*y+2)**2)
)

CURVATURE_FORMULAS_PASS = all([
    sp.simplify(Ricci_scalar-R_expected)==0,
    sp.simplify(Ricci_sq-Ricci_sq_expected)==0,
    sp.simplify(Kretschmann-K_expected)==0,
])
assert CURVATURE_FORMULAS_PASS

print("R =", Ricci_scalar)
print("Ricci^2 =", Ricci_sq)
print("Kretschmann =", Kretschmann)
print("CURVATURE_FORMULAS_PASS =", CURVATURE_FORMULAS_PASS)

R = -2*c*y**2/(r**2*(c*y**2 + 4*y + 2))
Ricci^2 = 4*c**2*y**4/(r**4*(c*y**2 + 4*y + 2)**2)
Kretschmann = 4*y**2*(3*c**2*y**2 + 16*c*y + 48)/(r**4*(c*y**2 + 4*y + 2)**2)
CURVATURE_FORMULAS_PASS = True


Pour \(0<c<2\), `.3.3.18` donne un rayon d'aire limite fini \(r_{\min}>0\) lorsque \(y\to\infty\).

Les invariants ont alors les limites :

\[
\boxed{
R\to-\frac{2}{r_{\min}^2}
}
\]

\[
\boxed{
R_{\mu\nu}R^{\mu\nu}
\to\frac{4}{r_{\min}^4}
}
\]

\[
\boxed{
R_{\mu\nu\rho\sigma}R^{\mu\nu\rho\sigma}
\to\frac{12}{r_{\min}^4}.
}
\]

La divergence de \(A^2\) en coordonnée d'aire n'est donc pas une singularité de courbure.

In [6]:
# PPN19.9 — Inner curvature limits
rmin = sp.symbols("r_min", positive=True)

R_inner = sp.simplify(sp.limit(Ricci_scalar.subs(r,rmin),y,sp.oo))
Ricci_sq_inner = sp.simplify(sp.limit(Ricci_sq.subs(r,rmin),y,sp.oo))
K_inner = sp.simplify(sp.limit(Kretschmann.subs(r,rmin),y,sp.oo))

INNER_RICCI_SCALAR_FINITE = (R_inner == -2/rmin**2)
INNER_RICCI_SQ_FINITE = (Ricci_sq_inner == 4/rmin**4)
INNER_KRETSCHMANN_FINITE = (K_inner == 12/rmin**4)

INNER_CURVATURE_REGULARITY_PASS = all([
    INNER_RICCI_SCALAR_FINITE,
    INNER_RICCI_SQ_FINITE,
    INNER_KRETSCHMANN_FINITE,
])
assert INNER_CURVATURE_REGULARITY_PASS

print("R ->", R_inner)
print("Ricci^2 ->", Ricci_sq_inner)
print("K ->", K_inner)
print("INNER_CURVATURE_REGULARITY_PASS =", INNER_CURVATURE_REGULARITY_PASS)

R -> -2/r_min**2
Ricci^2 -> 4/r_min**4
K -> 12/r_min**4
INNER_CURVATURE_REGULARITY_PASS = True


# PPN19.10 — Coordonnée propre et gorge locale

La variable \(y\) diverge à \(r_{\min}\), mais :

\[
z\equiv\frac1y
\]

est régulière.

Avec la distance radiale propre \(d\ell=A\,dr\), et :

\[
Q(z)=z^2+2z+\frac c2,
\]

le système devient localement :

\[
\boxed{
\frac{dr}{d\ell}=\frac{z}{\sqrt{Q(z)}}
}
\]

\[
\boxed{
\frac{dz}{d\ell}=\frac{\sqrt{Q(z)}}{r}
}
\]

\[
\boxed{
\frac{d\ln N}{d\ell}
=
\frac{1}{r\sqrt{Q(z)}}.
}
\]

À \(z=0\) :

\[
\frac{dr}{d\ell}=0,
\qquad
\frac{d^2r}{d\ell^2}=\frac1{r_{\min}}>0.
\]

La surface est donc une **gorge locale régulière** dans le secteur audité.

Cela ne prouve pas encore une extension globale complète : pour \(0<c<2\), \(Q(z)\) possède des racines négatives sur le côté prolongé.

In [7]:
# PPN19.11 — Local proper-distance extension at the inner surface
z = sp.symbols("z", real=True)
Qz = sp.factor(z**2+2*z+c/2)

dr_dl = z/sp.sqrt(Qz)
dz_dl = sp.sqrt(Qz)/rmin
dlogN_dl = 1/(rmin*sp.sqrt(Qz))

dr_dl_at_throat = sp.simplify(dr_dl.subs(z,0))
d2r_dl2_at_throat = sp.simplify(
    sp.diff(dr_dl,z).subs(z,0) * dz_dl.subs(z,0)
)
dlogN_dl_at_throat = sp.simplify(dlogN_dl.subs(z,0))

LOCAL_THROAT_FIRST_DERIVATIVE_PASS = (dr_dl_at_throat == 0)
LOCAL_THROAT_SECOND_DERIVATIVE_PASS = (d2r_dl2_at_throat == 1/rmin)
# Inherited route-D domain is 0<c<2.  Compare algebraically without
# asking SymPy to choose a square-root branch for a merely real symbol c.
LOCAL_THROAT_LAPSE_DERIVATIVE_FINITE = (
    sp.simplify(dlogN_dl_at_throat**2 - 2/(c*rmin**2)) == 0
)

LOCAL_INNER_THROAT_EXTENSION_PASS = all([
    LOCAL_THROAT_FIRST_DERIVATIVE_PASS,
    LOCAL_THROAT_SECOND_DERIVATIVE_PASS,
    LOCAL_THROAT_LAPSE_DERIVATIVE_FINITE,
])

GLOBAL_TWO_SIDED_EXTENSION_CLASSIFIED = False

assert LOCAL_INNER_THROAT_EXTENSION_PASS

print("Q(z) =", Qz)
print("dr/dl | throat =", dr_dl_at_throat)
print("d2r/dl2 | throat =", d2r_dl2_at_throat)
print("d ln N/dl | throat =", dlogN_dl_at_throat)
print("LOCAL_INNER_THROAT_EXTENSION_PASS =", LOCAL_INNER_THROAT_EXTENSION_PASS)
print("GLOBAL_TWO_SIDED_EXTENSION_CLASSIFIED =", GLOBAL_TWO_SIDED_EXTENSION_CLASSIFIED)

Q(z) = (c + 2*z**2 + 4*z)/2
dr/dl | throat = 0
d2r/dl2 | throat = 1/r_min
d ln N/dl | throat = sqrt(2)/(sqrt(c)*r_min)
LOCAL_INNER_THROAT_EXTENSION_PASS = True
GLOBAL_TWO_SIDED_EXTENSION_CLASSIFIED = False


# PPN19.12 — Transformation isotrope

`.3.3.18` donne, avec \(x=M/r\) :

\[
N^2
=
1-2x-\frac c6x^3+O(x^4),
\]

\[
A^2
=
1+2x+
\left(4+\frac c2\right)x^2+
\left(8+\frac{5c}{2}\right)x^3+\cdots.
\]

On introduit le rayon isotrope \(\rho\) :

\[
A^2dr^2+r^2d\Omega^2
=
B(\rho)^2
(d\rho^2+\rho^2d\Omega^2),
\]

et :

\[
s\equiv\frac M\rho.
\]

La relation différentielle est :

\[
\frac{d\ln s}{d\ln x}=A(x).
\]

In [8]:
# PPN19.13 — Isotropic series and PPN gamma/beta
s = sp.symbols("s", positive=True)
a2,a3,a4 = sp.symbols("a2 a3 a4")

x_ansatz = s+a2*s**2+a3*s**3+a4*s**4

A2_x = (
    1+2*x_ansatz
    +(4+c/sp.Integer(2))*x_ansatz**2
    +(8+5*c/sp.Integer(2))*x_ansatz**3
)
A_x = sp.series(sp.sqrt(A2_x),s,0,4).removeO()

# d ln s / d ln x = A  <=>  dx/ds = x/(s A)
iso_eq = sp.series(
    sp.diff(x_ansatz,s)-x_ansatz/(s*A_x),
    s,0,4
).removeO().expand()

sol_a2 = sp.solve(sp.Eq(iso_eq.coeff(s,1),0),a2)[0]
sol_a3 = sp.solve(
    sp.Eq(iso_eq.subs(a2,sol_a2).coeff(s,2),0),a3
)[0]
sol_a4 = sp.solve(
    sp.Eq(iso_eq.subs({a2:sol_a2,a3:sol_a3}).coeff(s,3),0),a4
)[0]

x_s = sp.expand(x_ansatz.subs({
    a2:sol_a2,a3:sol_a3,a4:sol_a4
}))

N2_s = sp.series(
    1-2*x_s-c*x_s**3/sp.Integer(6),
    s,0,4
).removeO().expand()

B2_s = sp.series((s/x_s)**2,s,0,3).removeO().expand()

# g00=-N^2 = -1+2U-2 beta U^2+...
g00_s = sp.expand(-N2_s)
beta = sp.simplify(-g00_s.coeff(s,2)/2)

# gij=(1+2 gamma U+...) delta_ij
gamma = sp.simplify(B2_s.coeff(s,1)/2)

PPN_GAMMA_ONE_PASS = (gamma == 1)
PPN_BETA_ONE_PASS = (beta == 1)
STANDARD_PPN_GAMMA_BETA_CLASSIFIED = (
    PPN_GAMMA_ONE_PASS and PPN_BETA_ONE_PASS
)
assert STANDARD_PPN_GAMMA_BETA_CLASSIFIED

print("x(s) =", x_s)
print("g00 =", g00_s)
print("B^2 =", B2_s)
print("gamma =", gamma)
print("beta =", beta)
print("STANDARD_PPN_GAMMA_BETA_CLASSIFIED =", STANDARD_PPN_GAMMA_BETA_CLASSIFIED)

x(s) = c*s**4/6 - c*s**3/8 - s**4/2 + 3*s**3/4 - s**2 + s
g00 = -c*s**3/12 + 3*s**3/2 - 2*s**2 + 2*s - 1
B^2 = c*s**2/4 + 3*s**2/2 + 2*s + 1
gamma = 1
beta = 1
STANDARD_PPN_GAMMA_BETA_CLASSIFIED = True


Le résultat est :

\[
\boxed{\gamma_{\rm GVH}=1},
\qquad
\boxed{\beta_{\rm GVH}=1}
\]

pour la solution sphérique route D au niveau PPN standard.

La déformation \(c=c_1+c_4\) apparaît seulement aux ordres supérieurs :

\[
\boxed{
g_{00}
=
-1+2s-2s^2+
\left(\frac32-\frac{c}{12}\right)s^3+\cdots
}
\]

et :

\[
\boxed{
g_{ij}
=
\left[
1+2s+
\left(\frac32+\frac c4\right)s^2+\cdots
\right]\delta_{ij}.
}
\]

Donc Cassini-\(\gamma\) et le \(\beta\) PPN standard ne fournissent **pas**, à eux seuls, une borne directe sur \(c\) dans ce secteur statique.

Les paramètres \(\alpha_1,\alpha_2\) ne peuvent pas être extraits d'une solution statique sphérique : ils nécessitent un système en mouvement par rapport au champ vectoriel préféré.

In [9]:
# PPN19.14 — Higher-order coefficients and coupling-separation ledger
g00_U3_coeff = sp.factor(g00_s.coeff(s,3))
gij_U2_coeff = sp.factor(B2_s.coeff(s,2))

c13 = sp.symbols("c13", real=True)
c14 = c

TENSOR_SPEED_SQUARED_INHERITED = 1/(1-c13)
ROUTE_D_STRONG_FIELD_COUPLING = c14

PREFERRED_FRAME_ALPHA1_CLASSIFIED = False
PREFERRED_FRAME_ALPHA2_CLASSIFIED = False
REAL_DATA_BOUND_ON_C14_AUTHORIZED = False

print("g00 U^3 coefficient =", g00_U3_coeff)
print("gij U^2 coefficient =", gij_U2_coeff)
print("route-D coupling c14 = c1+c4")
print("tensor-speed coupling c13 = c1+c3")
print("c_T^2 inherited =", TENSOR_SPEED_SQUARED_INHERITED)
print("PREFERRED_FRAME_ALPHA1_CLASSIFIED =", PREFERRED_FRAME_ALPHA1_CLASSIFIED)
print("PREFERRED_FRAME_ALPHA2_CLASSIFIED =", PREFERRED_FRAME_ALPHA2_CLASSIFIED)
print("REAL_DATA_BOUND_ON_C14_AUTHORIZED =", REAL_DATA_BOUND_ON_C14_AUTHORIZED)

g00 U^3 coefficient = -(c - 18)/12
gij U^2 coefficient = (c + 6)/4
route-D coupling c14 = c1+c4
tensor-speed coupling c13 = c1+c3
c_T^2 inherited = 1/(1 - c13)
PREFERRED_FRAME_ALPHA1_CLASSIFIED = False
PREFERRED_FRAME_ALPHA2_CLASSIFIED = False
REAL_DATA_BOUND_ON_C14_AUTHORIZED = False


# PPN19.15 — Discipline des couplages

Deux combinaisons doivent rester séparées :

\[
\boxed{
c_{14}\equiv c_1+c_4
}
\]

contrôle la déformation sphérique route D, tandis que le secteur tensoriel linéarisé hérité donne :

\[
\boxed{
c_T^2=\frac1{1-c_{13}},
\qquad
c_{13}\equiv c_1+c_3.
}
\]

Une contrainte observationnelle sur \(c_T\) ne peut donc pas être convertie directement en borne sur \(c_{14}\) sans relation supplémentaire entre les couplages.

Aucune telle relation n'est introduite ici.

In [10]:
# PPN19.16 — Final scientific classifier
FULL_SPHERICALLY_SYMMETRIC_VARIATIONAL_EOM_CROSSCHECK_PASS = all([
    FULL_SPHERICAL_METRIC_EOMS_MATERIALIZED,
    FULL_SPHERICAL_METRIC_EOM_CROSSCHECK_PASS,
    ALIGNED_VECTOR_FIRST_VARIATION_ZERO,
])

INNER_LOCAL_GEOMETRY_CLASSIFIED = all([
    INNER_CURVATURE_REGULARITY_PASS,
    LOCAL_INNER_THROAT_EXTENSION_PASS,
])

DEFORMED_SPHERICAL_ROUTE_D_EXTERIOR_EOM_PASS = (
    FULL_SPHERICALLY_SYMMETRIC_VARIATIONAL_EOM_CROSSCHECK_PASS
)

DEFORMED_SPHERICAL_ROUTE_D_LOCAL_INNER_REGULARITY_PASS = (
    INNER_LOCAL_GEOMETRY_CLASSIFIED
)

DEFORMED_SPHERICAL_ROUTE_D_STANDARD_PPN_PASS = (
    STANDARD_PPN_GAMMA_BETA_CLASSIFIED
    and gamma==1 and beta==1
)

# Scope-preserving locks
FULL_ARBITRARY_4D_COVARIANT_TENSOR_EOM_INDEPENDENTLY_CROSSCHECKED = False
GLOBAL_TWO_SIDED_EXTENSION_PASS = False
PREFERRED_FRAME_PPN_CLASSIFIED = False

EXACT_SCHWARZSCHILD_BENCHMARK_PASS = False
SCHWARZSCHILD_BENCHMARK_AUTHORIZED = False

DEFORMED_SPHERICAL_GVH_BENCHMARK_AUTHORIZED = True
DEFORMED_SPHERICAL_GVH_BENCHMARK_PASS = False
# Not promoted to full PASS until global extension and independent full-4D tensor crosscheck are closed.

KERR_BENCHMARK_AUTHORIZED = False
REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED = False
QUANTIZATION_READY = False

PPN19_OBSTRUCTIONS = [
    "GLOBAL-TWO-SIDED-INNER-EXTENSION-NOT-YET-CLASSIFIED",
    "FULL-ARBITRARY-4D-COVARIANT-TENSOR-EOM-NOT-INDEPENDENTLY-CROSSCHECKED",
    "PREFERRED-FRAME-PPN-ALPHA1-ALPHA2-NOT-YET-DERIVED",
    "REAL-DATA-BOUND-ON-c14-NOT-AUTHORIZED",
]

PPN19_LOCAL_AUDIT_PASS = all([
    UPSTREAM_GATE,
    FULL_SPHERICALLY_SYMMETRIC_VARIATIONAL_EOM_CROSSCHECK_PASS,
    INNER_CURVATURE_REGULARITY_PASS,
    LOCAL_INNER_THROAT_EXTENSION_PASS,
    STANDARD_PPN_GAMMA_BETA_CLASSIFIED,
    not EXACT_SCHWARZSCHILD_BENCHMARK_PASS,
    DEFORMED_SPHERICAL_GVH_BENCHMARK_AUTHORIZED,
    not DEFORMED_SPHERICAL_GVH_BENCHMARK_PASS,
    not KERR_BENCHMARK_AUTHORIZED,
    not REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED,
])

PPN19_NEXT_AUTHORIZED = (
    "AUDIT-DEFORMED-SPHERICAL-GVH-GLOBAL-INNER-EXTENSION-FULL-4D-COVARIANT-CROSSCHECK-AND-2PN-OBSERVABLES"
    if PPN19_LOCAL_AUDIT_PASS
    else "REPAIR-.3.3.19-FULL-SPHERICAL-EOM-INNER-PPN-AUDIT"
)

assert PPN19_LOCAL_AUDIT_PASS

print("FULL_SPHERICALLY_SYMMETRIC_VARIATIONAL_EOM_CROSSCHECK_PASS =",
      FULL_SPHERICALLY_SYMMETRIC_VARIATIONAL_EOM_CROSSCHECK_PASS)
print("DEFORMED_SPHERICAL_ROUTE_D_EXTERIOR_EOM_PASS =",
      DEFORMED_SPHERICAL_ROUTE_D_EXTERIOR_EOM_PASS)
print("INNER_CURVATURE_REGULARITY_PASS =", INNER_CURVATURE_REGULARITY_PASS)
print("LOCAL_INNER_THROAT_EXTENSION_PASS =", LOCAL_INNER_THROAT_EXTENSION_PASS)
print("STANDARD_PPN_GAMMA_BETA_CLASSIFIED =", STANDARD_PPN_GAMMA_BETA_CLASSIFIED)
print("PPN_GAMMA =", gamma)
print("PPN_BETA =", beta)
print("DEFORMED_SPHERICAL_GVH_BENCHMARK_PASS =", DEFORMED_SPHERICAL_GVH_BENCHMARK_PASS)
print("KERR_BENCHMARK_AUTHORIZED =", KERR_BENCHMARK_AUTHORIZED)
print("REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED =", REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED)
print("PPN19_OBSTRUCTIONS =", PPN19_OBSTRUCTIONS)
print("PPN19_NEXT_AUTHORIZED =", PPN19_NEXT_AUTHORIZED)

FULL_SPHERICALLY_SYMMETRIC_VARIATIONAL_EOM_CROSSCHECK_PASS = True
DEFORMED_SPHERICAL_ROUTE_D_EXTERIOR_EOM_PASS = True
INNER_CURVATURE_REGULARITY_PASS = True
LOCAL_INNER_THROAT_EXTENSION_PASS = True
STANDARD_PPN_GAMMA_BETA_CLASSIFIED = True
PPN_GAMMA = 1
PPN_BETA = 1
DEFORMED_SPHERICAL_GVH_BENCHMARK_PASS = False
KERR_BENCHMARK_AUTHORIZED = False
REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED = False
PPN19_OBSTRUCTIONS = ['GLOBAL-TWO-SIDED-INNER-EXTENSION-NOT-YET-CLASSIFIED', 'FULL-ARBITRARY-4D-COVARIANT-TENSOR-EOM-NOT-INDEPENDENTLY-CROSSCHECKED', 'PREFERRED-FRAME-PPN-ALPHA1-ALPHA2-NOT-YET-DERIVED', 'REAL-DATA-BOUND-ON-c14-NOT-AUTHORIZED']
PPN19_NEXT_AUTHORIZED = AUDIT-DEFORMED-SPHERICAL-GVH-GLOBAL-INNER-EXTENSION-FULL-4D-COVARIANT-CROSSCHECK-AND-2PN-OBSERVABLES


# PPN19.17 — Verdict de portée

Ce notebook ferme, dans le secteur sphérique audité :

\[
\boxed{
E_N=E_A=E_R=E_\eta=0
}
\]

pour la solution route D, avec variation de \(R(r)\) effectuée avant la gauge \(R=r\).

Il montre également :

\[
\boxed{
R,\quad R_{\mu\nu}R^{\mu\nu},\quad
R_{\mu\nu\rho\sigma}R^{\mu\nu\rho\sigma}
}
\]

finis à \(r_{\min}\), et matérialise une gorge locale en distance propre.

Enfin :

\[
\boxed{\gamma=\beta=1}
\]

au PPN standard.

Mais les points suivants restent volontairement ouverts :

- extension globale de l'autre côté de la gorge ;
- cross-check tensoriel 4D indépendant hors réduction sphérique ;
- \(\alpha_1,\alpha_2\) et autres paramètres de cadre préféré ;
- observables 2PN/fort champ ;
- contraintes par données réelles.

Aucun de ces points n'est promu par inférence.

In [11]:
# PPN19.18 — Machine-readable artifact
artifact = {
    "notebook": "GVH_Diagonal_Cubic_0.3.2.7.3.7.3.3.19_Deformed_Spherical_GVH_Full_Spherical_EOM_Inner_Regularity_and_PPN_Audit_FAST",
    "execution_scope": "DEFORMED_SPHERICAL_FULL_SPHERICAL_EOM_INNER_REGULARITY_PPN",
    "upstream": UPSTREAM,
    "spherical_eom": {
        "varied_before_areal_gauge": ["N(r)", "A(r)", "R(r)"],
        "aligned_vector_first_variation_zero": ALIGNED_VECTOR_FIRST_VARIATION_ZERO,
        "metric_eom_crosscheck_pass": FULL_SPHERICAL_METRIC_EOM_CROSSCHECK_PASS,
        "full_spherically_symmetric_variational_eom_crosscheck_pass":
            FULL_SPHERICALLY_SYMMETRIC_VARIATIONAL_EOM_CROSSCHECK_PASS,
        "route_D_equations": {
            "A_squared": "P(y)=1+2y+(c/2)y^2",
            "y_ode": "r y'=-y P(y)",
        },
    },
    "inner_geometry": {
        "Ricci_scalar": str(Ricci_scalar),
        "Ricci_squared": str(Ricci_sq),
        "Kretschmann": str(Kretschmann),
        "inner_limits": {
            "R": str(R_inner),
            "Ricci_squared": str(Ricci_sq_inner),
            "Kretschmann": str(K_inner),
        },
        "inner_curvature_regularity_pass": INNER_CURVATURE_REGULARITY_PASS,
        "local_throat_extension_pass": LOCAL_INNER_THROAT_EXTENSION_PASS,
        "global_two_sided_extension_classified": GLOBAL_TWO_SIDED_EXTENSION_CLASSIFIED,
    },
    "ppn": {
        "isotropic_x_of_s": str(x_s),
        "g00": str(g00_s),
        "B_squared": str(B2_s),
        "gamma": str(gamma),
        "beta": str(beta),
        "standard_ppn_gamma_beta_classified": STANDARD_PPN_GAMMA_BETA_CLASSIFIED,
        "preferred_frame_alpha1_classified": PREFERRED_FRAME_ALPHA1_CLASSIFIED,
        "preferred_frame_alpha2_classified": PREFERRED_FRAME_ALPHA2_CLASSIFIED,
    },
    "coupling_ledger": {
        "route_D_strong_field": "c14=c1+c4",
        "tensor_speed": "c13=c1+c3",
        "tensor_speed_squared_inherited": "1/(1-c13)",
        "direct_c13_to_c14_identification_authorized": False,
    },
    "scientific_status": {
        "DEFORMED_SPHERICAL_ROUTE_D_EXTERIOR_EOM_PASS":
            DEFORMED_SPHERICAL_ROUTE_D_EXTERIOR_EOM_PASS,
        "DEFORMED_SPHERICAL_ROUTE_D_LOCAL_INNER_REGULARITY_PASS":
            DEFORMED_SPHERICAL_ROUTE_D_LOCAL_INNER_REGULARITY_PASS,
        "DEFORMED_SPHERICAL_ROUTE_D_STANDARD_PPN_PASS":
            DEFORMED_SPHERICAL_ROUTE_D_STANDARD_PPN_PASS,
        "FULL_ARBITRARY_4D_COVARIANT_TENSOR_EOM_INDEPENDENTLY_CROSSCHECKED":
            FULL_ARBITRARY_4D_COVARIANT_TENSOR_EOM_INDEPENDENTLY_CROSSCHECKED,
        "GLOBAL_TWO_SIDED_EXTENSION_PASS": GLOBAL_TWO_SIDED_EXTENSION_PASS,
        "PREFERRED_FRAME_PPN_CLASSIFIED": PREFERRED_FRAME_PPN_CLASSIFIED,
        "EXACT_SCHWARZSCHILD_BENCHMARK_PASS": EXACT_SCHWARZSCHILD_BENCHMARK_PASS,
        "DEFORMED_SPHERICAL_GVH_BENCHMARK_AUTHORIZED":
            DEFORMED_SPHERICAL_GVH_BENCHMARK_AUTHORIZED,
        "DEFORMED_SPHERICAL_GVH_BENCHMARK_PASS":
            DEFORMED_SPHERICAL_GVH_BENCHMARK_PASS,
        "KERR_BENCHMARK_AUTHORIZED": KERR_BENCHMARK_AUTHORIZED,
        "REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED":
            REAL_DATA_PARAMETER_INFERENCE_AUTHORIZED,
        "QUANTIZATION_READY": QUANTIZATION_READY,
    },
    "verdict": {
        "PPN19_LOCAL_AUDIT_PASS": PPN19_LOCAL_AUDIT_PASS,
        "obstructions": PPN19_OBSTRUCTIONS,
    },
    "next_authorized": PPN19_NEXT_AUTHORIZED,
    "scope_note": "PASS is local/spherical: independent spherical metric equations and aligned vector first variation close; inner curvature is finite and a local throat extension exists; gamma=beta=1 at standard PPN. Full arbitrary 4D tensor crosscheck, global second-side extension, preferred-frame PPN and real-data inference remain locked."
}

export_dir = Path("/content/gvh_exports") if Path("/content").exists() else Path("/mnt/data")
export_dir.mkdir(parents=True, exist_ok=True)
artifact_path = export_dir / "gvh_0.3.2.7.3.7.3.3.19_Deformed_Spherical_GVH_Full_Spherical_EOM_Inner_Regularity_and_PPN_Audit_FAST.json"
artifact_path.write_text(json.dumps(artifact, indent=2, ensure_ascii=False), encoding="utf-8")
print("PPN19 artifact =", artifact_path)

PPN19 artifact = /content/gvh_exports/gvh_0.3.2.7.3.7.3.3.19_Deformed_Spherical_GVH_Full_Spherical_EOM_Inner_Regularity_and_PPN_Audit_FAST.json


# Conclusion

`.3.3.19` transforme la route D de simple candidat réduit en solution **sphériquement auto-cohérente dans le scope variationnel testé** :

\[
\boxed{
E_N=E_A=E_R=E_\eta=0.
}
\]

La surface intérieure vérifie :

\[
\boxed{
R\to-\frac2{r_{\min}^2},\quad
R_{\mu\nu}R^{\mu\nu}\to\frac4{r_{\min}^4},\quad
K\to\frac{12}{r_{\min}^4}
}
\]

et possède une gorge locale régulière en distance propre.

Au faible champ isotrope :

\[
\boxed{\gamma=1,\qquad\beta=1}.
\]

La première déformation statique apparaît au-delà du PPN standard dominant et dépend de :

\[
\boxed{c_{14}=c_1+c_4}.
\]

Cela ne doit pas être confondu avec :

\[
\boxed{c_{13}=c_1+c_3}
\]

qui contrôle la vitesse tensorielle héritée.

La prochaine marche reste un audit de fermeture globale/4D et d'observables 2PN, pas encore une inférence sur données.